# 00 — Prepare Nigeria Data (Instructor)

**Repository:** `zubairgis/nigeria-population-demography-course`  
**Stage:** validated pilot first; national processing is blocked until the pilot passes.

**Pilot:** Bayelsa State → Sagbama LGA.

This notebook downloads only documented sources, preserves source identifiers, performs fractional-overlap population aggregation, records unmatched settlement names, and packages the pilot outputs. It does **not** fabricate replacement data when a source fails.

In [ ]:
# Cell P1 — Install preparation libraries
!pip -q install geopandas pyogrio shapely pyproj rasterio exactextract requests tqdm pyarrow folium
print("Preparation environment ready.")

In [ ]:
# Cell P2 — Clone the course repository
from pathlib import Path
import os, subprocess

REPO_URL = "https://github.com/zubairgis/nigeria-population-demography-course.git"
REPO_DIR = Path("/content/nigeria-population-demography-course")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
print("Repository:", REPO_DIR)

In [ ]:
# Cell P3 — Inspect the data manifest before downloading
import pandas as pd
manifest = pd.read_csv(REPO_DIR / "config/dataset_manifest.csv")
display(manifest[["dataset_id","version","reference_year","licence","status"]])
unresolved = manifest[manifest["status"].astype(str).str.contains("unresolved|partial", case=False, regex=True)]
print(f"Unresolved/partial entries: {len(unresolved)}")
print("Do not publicly redistribute any dataset whose exact-version licence remains unresolved.")

In [ ]:
# Cell P4 — Verify live source endpoints without downloading data
import subprocess, sys
for ds in ["state_boundaries","lga_boundaries","settlement_names","settlement_extents","health_v2","worldpop_core"]:
    print("\n---", ds, "---")
    subprocess.run([
        sys.executable, str(REPO_DIR/"scripts/01_download_sources.py"),
        "--dataset", ds, "--metadata-only"
    ], check=True)

## Stop point

If Cell P4 fails, stop. Do not replace the source with guessed URLs or fabricated sample data. Check the official source and update the endpoint registry/manifest first.

In [ ]:
# Cell P5 — Download national State and LGA boundary layers
DOWNLOADS = REPO_DIR / "downloads"
DOWNLOADS.mkdir(exist_ok=True)
for ds in ["state_boundaries","lga_boundaries"]:
    subprocess.run([
        sys.executable, str(REPO_DIR/"scripts/01_download_sources.py"),
        "--dataset", ds, "--output", str(DOWNLOADS)
    ], check=True)
print("Boundary downloads complete.")

In [ ]:
# Cell P6 — Validate boundaries and create State/LGA lookup tables
subprocess.run([
    sys.executable, str(REPO_DIR/"scripts/02_validate_boundaries.py"),
    "--states", str(DOWNLOADS/"state_boundaries.geojson"),
    "--lgas", str(DOWNLOADS/"lga_boundaries.geojson"),
    "--out-dir", str(REPO_DIR/"data/lookups")
], check=True)

states = pd.read_csv(REPO_DIR/"data/lookups/states.csv", dtype=str)
lgas = pd.read_csv(REPO_DIR/"data/lookups/lgas.csv", dtype=str)
print("States/FCT:", len(states))
print("LGA/Area Council records:", len(lgas))
display(lgas[lgas.state_name.str.contains("FCT|Abuja", case=False, regex=True, na=False)])

In [ ]:
# Cell P7 — Select the real pilot by source attributes and save its analytical boundary
import geopandas as gpd

PILOT_STATE = "Bayelsa"
PILOT_LGA = "Sagbama"
PILOT_DIR = Path("/content/pilot_sagbama")
PILOT_DIR.mkdir(parents=True, exist_ok=True)

all_lgas = gpd.read_file(DOWNLOADS/"lga_boundaries.geojson").to_crs(4326)
pilot = all_lgas[(all_lgas["statename"].str.casefold()==PILOT_STATE.casefold()) &
                 (all_lgas["lganame"].str.casefold()==PILOT_LGA.casefold())].copy()
if len(pilot) != 1:
    raise ValueError(f"Expected exactly one {PILOT_STATE}/{PILOT_LGA} record, found {len(pilot)}")
if not pilot.geometry.iloc[0].is_valid:
    raise ValueError("Pilot LGA geometry is invalid.")

PILOT_LGA_ID = str(pilot.iloc[0]["lgacode"])
PILOT_STATE_ID = str(pilot.iloc[0]["statecode"])
PILOT_LGA_FILE = PILOT_DIR/"sagbama_lga.gpkg"
pilot.to_file(PILOT_LGA_FILE, driver="GPKG")
BBOX = tuple(float(x) for x in pilot.total_bounds)
print("State code:", PILOT_STATE_ID)
print("LGA code:", PILOT_LGA_ID)
print("WGS84 bbox:", BBOX)
print("Saved:", PILOT_LGA_FILE)

In [ ]:
# Cell P8 — Download only pilot-area settlement, name and health-facility records
bbox_args = [str(v) for v in BBOX]
for ds in ["settlement_extents","settlement_names","health_v2"]:
    print("\nDownloading", ds)
    cmd = [sys.executable, str(REPO_DIR/"scripts/01_download_sources.py"),
           "--dataset", ds, "--output", str(PILOT_DIR), "--bbox", *bbox_args]
    subprocess.run(cmd, check=True)
print("Pilot-area vector downloads complete.")

In [ ]:
# Cell P9 — Clip settlement blocks to Sagbama and match documented settlement names
SETTLEMENTS_GPKG = PILOT_DIR/"settlement_components.gpkg"
subprocess.run([
    sys.executable, str(REPO_DIR/"scripts/04_prepare_settlements.py"),
    "--extents", str(PILOT_DIR/"settlement_extents.geojson"),
    "--names", str(PILOT_DIR/"settlement_names.geojson"),
    "--lga", str(PILOT_LGA_FILE),
    "--lga-id", PILOT_LGA_ID,
    "--out", str(SETTLEMENTS_GPKG)
], check=True)
sett = gpd.read_file(SETTLEMENTS_GPKG)
print("Settlement components:", len(sett))
print("Name matching:")
print(sett["name_match_status"].value_counts(dropna=False))
print("GRID3-reported building_count available:", "building_count" in sett.columns)

In [ ]:
# Cell P10 — Prepare health facilities inside the selected LGA
HEALTH_GPKG = PILOT_DIR/"health_facilities_sagbama.gpkg"
subprocess.run([
    sys.executable, str(REPO_DIR/"scripts/06_prepare_health_facilities.py"),
    "--facilities", str(PILOT_DIR/"health_v2.geojson"),
    "--lga", str(PILOT_LGA_FILE),
    "--out", str(HEALTH_GPKG)
], check=True)
health = gpd.read_file(HEALTH_GPKG)
print("Health-facility records inside Sagbama:", len(health))
display(health.head())

## Detected building footprints

For the pilot, building counts are derived from **Google Open Buildings V3 polygons**, not from an invented occupancy factor and not by copying a whole GRID3 block count into a clipped cross-LGA component.

The downloader follows Google's regional S2-shard method and uses Google's tile-specific **90% precision threshold**. Counts remain counts of detected footprints only.

In [ ]:
# Cell P10B — Download Google Open Buildings V3 footprints for Sagbama
OPEN_BUILDINGS_FILE = PILOT_DIR/"google_open_buildings_sagbama.parquet"
OPEN_BUILDINGS_DOWNLOAD_SUMMARY = PILOT_DIR/"open_buildings_download_summary.json"

subprocess.run([
    sys.executable, str(REPO_DIR/"scripts/05_download_open_buildings.py"),
    "--aoi", str(PILOT_LGA_FILE),
    "--out", str(OPEN_BUILDINGS_FILE),
    "--summary", str(OPEN_BUILDINGS_DOWNLOAD_SUMMARY),
    "--precision", "90"
], check=True)

import json
ob_download = json.loads(OPEN_BUILDINGS_DOWNLOAD_SUMMARY.read_text())
print("Detected footprints retained:", ob_download["detected_footprints_output"])
print("S2 level-6 tiles considered:", ob_download["tiles_considered"])
print("Downloaded bytes:", ob_download["download_bytes"])

In [ ]:
# Cell P10C — Allocate each detected footprint to at most one settlement component
BUILDING_COUNTS_CSV = PILOT_DIR/"settlement_building_counts.csv"
BUILDING_ALLOCATION_SUMMARY = PILOT_DIR/"building_allocation_summary.json"

subprocess.run([
    sys.executable, str(REPO_DIR/"scripts/05_count_buildings.py"),
    "--buildings", str(OPEN_BUILDINGS_FILE),
    "--zones", str(SETTLEMENTS_GPKG),
    "--zone-id", "settlement_component_id",
    "--out", str(BUILDING_COUNTS_CSV),
    "--summary", str(BUILDING_ALLOCATION_SUMMARY)
], check=True)

building_counts = pd.read_csv(BUILDING_COUNTS_CSV)
building_allocation = json.loads(BUILDING_ALLOCATION_SUMMARY.read_text())
display(building_counts.head())
print("Allocated to mapped settlement components:",
      building_allocation["allocated_to_one_component"])
print("Inside Sagbama but outside mapped settlement components:",
      building_allocation["inside_lga_but_outside_mapped_settlement_components"])

## Population download

The six required WorldPop rasters are each about 149–150 MB, so the pilot download is roughly **0.9 GB**, not the full 8.56 GB age-sex archive. The required layers are from the same 2025 R2025A v1 constrained 100 m family.

- total = male total + female total
- under-1 = male 0–12 months + female 0–12 months
- under-5 = under-1 + male 1–4 + female 1–4

Under-1 is already included in under-5.

In [ ]:
# Cell P11 — Download only the six WorldPop rasters required by this course
WORLDPOP_DIR = PILOT_DIR/"worldpop_2025"
subprocess.run([
    sys.executable, str(REPO_DIR/"scripts/01_download_sources.py"),
    "--dataset", "worldpop_core",
    "--output", str(WORLDPOP_DIR)
], check=True)
print("WorldPop core download complete.")

In [ ]:
# Cell P12 — Aggregate WorldPop to the LGA using fractional cell overlap
LGA_POP_CSV = PILOT_DIR/"lga_population_summary.csv"
subprocess.run([
    sys.executable, str(REPO_DIR/"scripts/03_prepare_population.py"),
    "--dir", str(WORLDPOP_DIR),
    "--zones", str(PILOT_LGA_FILE),
    "--id-field", "lgacode",
    "--out", str(LGA_POP_CSV)
], check=True)
lga_pop = pd.read_csv(LGA_POP_CSV)
display(lga_pop)

In [ ]:
# Cell P13 — Aggregate the same demographic rasters to settlement components
SETT_POP_CSV = PILOT_DIR/"settlement_population.csv"
subprocess.run([
    sys.executable, str(REPO_DIR/"scripts/03_prepare_population.py"),
    "--dir", str(WORLDPOP_DIR),
    "--zones", str(SETTLEMENTS_GPKG),
    "--id-field", "settlement_component_id",
    "--out", str(SETT_POP_CSV)
], check=True)
sett_pop = pd.read_csv(SETT_POP_CSV)
print("Settlement population rows:", len(sett_pop))
display(sett_pop.head())

In [ ]:
# Cell P14 — Reconcile mapped-settlement population with the full LGA
metrics = ["population_total_est","population_male_est","population_female_est",
           "population_u1_est","population_u5_est"]
recon = []
for m in metrics:
    lga_value = float(lga_pop.loc[0, m])
    inside_value = float(sett_pop[m].sum(skipna=True))
    outside_value = lga_value - inside_value
    coverage_pct = (inside_value / lga_value * 100.0) if lga_value else float("nan")
    recon.append({"metric":m,"lga_estimate":lga_value,"inside_mapped_settlements":inside_value,
                  "outside_mapped_settlements":outside_value,"settlement_coverage_pct":coverage_pct})
recon = pd.DataFrame(recon)
RECON_CSV = PILOT_DIR/"population_reconciliation.csv"
recon.to_csv(RECON_CSV, index=False)
display(recon)
print("Settlement totals are not forced to equal the LGA total.")

In [ ]:
# Cell P15 — Build the settlement teaching table with actual footprint counts
sett_base = sett.drop(columns="geometry").copy()

# Preserve GRID3 whole-block counts only as provenance/diagnostic fields.
if "detected_building_count" in sett_base.columns:
    sett_base = sett_base.rename(
        columns={"detected_building_count": "grid3_full_block_count_provisional"}
    )

sett_table = (
    sett_base
    .merge(sett_pop, on="settlement_component_id", how="left", validate="one_to_one")
    .merge(building_counts, on="settlement_component_id", how="left", validate="one_to_one")
)
sett_table["detected_building_count"] = (
    sett_table["detected_building_count"].fillna(0).astype("int64")
)

keep = [c for c in [
    "source_settlement_id","settlement_component_id","lga_id","settlement_name",
    "settlement_name_ids","settlement_alt_names","name_match_status","extent_type",
    "detected_building_count","building_count_status","source_building_count",
    "grid3_full_block_count_provisional","component_fraction_of_source",
    "population_total_est","population_male_est","population_female_est",
    "population_u1_est","population_u5_est","population_reference_year",
    "population_dataset_id"
] if c in sett_table.columns]

SETT_TABLE_CSV = PILOT_DIR/"settlement_summary.csv"
sett_table[keep].to_csv(SETT_TABLE_CSV, index=False)
display(sett_table[keep].head(10))

print("Student-facing building count = Google Open Buildings V3 detected footprints")
print("allocated by source centroid at the documented confidence policy.")
print("No count is interpreted as households, occupied dwellings, or residential buildings.")


In [ ]:
# Cell P16 — Export a health-facility CSV without claiming operational status
health_csv = health.drop(columns="geometry").copy()
HEALTH_CSV = PILOT_DIR/"health_facilities.csv"
health_csv.to_csv(HEALTH_CSV, index=False)
print("Saved:", HEALTH_CSV)
print("No inference is made about whether a listed facility is currently operational, staffed, or providing a specific service.")

In [ ]:
# Cell P17 — Package the pilot outputs with file checksums
PILOT_ZIP = Path("/content/sagbama_population_demography_pilot.zip")
subprocess.run([
    sys.executable, str(REPO_DIR/"scripts/07_validate_and_package.py"),
    "--input-dir", str(PILOT_DIR),
    "--zip", str(PILOT_ZIP)
], check=True)
print("Pilot ZIP:", PILOT_ZIP)

## Pilot validation gate

Before any national processing or student-notebook development, confirm all of the following from the actual run:

1. State → LGA lookup integrity, including FCT Area Councils.
2. Valid Sagbama boundary and stable source codes.
3. Population aggregation from **people-per-grid-cell** rasters, not density.
4. Correct under-1 and under-5 definitions.
5. Settlement names classified as single, multiple or unmatched rather than forced.
6. Real Google Open Buildings detections downloaded with documented confidence thresholds, allocated once by source centroid, and reconciled between mapped settlements and the rest of the LGA.
7. Health facilities spatially assigned and deduplicated without operational-status claims.
8. Population inside and outside mapped settlement extents reported separately.
9. Pilot ZIP and checksums produced successfully.
10. Actual storage size and processing time recorded.

Only after these checks pass should nationwide state packages and the seven-chapter student notebook be built.